## 2022 Fifa World Cup Prediction Model

Amr Farahat

CMU / Tepper

2022-12-08

---

Most of the prediction work is already done by FIFA. The FIFA (ELO) ratings capture the recent history of national teams and provide a formula for calculating the win probability between any two teams:

References:

https://www.fifa.com/fifa-world-ranking/men?dateId=id13792

https://www.fifa.com/fifa-world-ranking/procedure-men

The followisng analysis is based on the FIFA October 2022 team point rankings. It simulates the winner starting from the Quarter_finals stage.

In [1]:
from numpy.random import default_rng
import pandas as pd

In [2]:
ratings = {"ARG": 1773.88, "NED": 1694.51, "BRA": 1841.3, "CRO": 1645.64, "ENG": 1728.47, "FRA": 1759.78, "POR": 1676.56, "MAR": 1563.5}

In [3]:
QF_list = ["NED","ARG","CRO","BRA","ENG","FRA","MAR","POR"]

In [4]:
num_matches = 4 + 2 + 1

In [5]:
def win_prob(team, opponent):
    team_rating = ratings[team]
    opponent_rating = ratings[opponent]
    rating_diff = team_rating - opponent_rating
    return(1 / (1 + 10**(-rating_diff/600)))

In [6]:
def pred_SF_list(QF_list, rng_list):
    # rng_list is a list of 4 uniform (between 0,1) random numbers
    
    team_1_win_prob = win_prob(QF_list[0], QF_list[1])
    team_3_win_prob = win_prob(QF_list[2], QF_list[3])
    team_5_win_prob = win_prob(QF_list[4], QF_list[5])
    team_7_win_prob = win_prob(QF_list[6], QF_list[7])
    
    SF_list = list()
    
    team_1_win_prob
    if rng_list[0] < team_1_win_prob:
        SF_list.append(QF_list[0])
    else:
        SF_list.append(QF_list[1])
        
    # ALternatively:
    #rng = default_rng()
    #rng.binomial(n=1, p=0.3)
    
    if rng_list[1] < team_3_win_prob:
        SF_list.append(QF_list[2])
    else:
        SF_list.append(QF_list[3])
    
    if rng_list[2] < team_5_win_prob:
        SF_list.append(QF_list[4])
    else:
        SF_list.append(QF_list[5])    
    
    if rng_list[3] < team_7_win_prob:
        SF_list.append(QF_list[6])
    else:
        SF_list.append(QF_list[7])    
    
    return(SF_list)

In [7]:
def pred_F_pair(SF_list, rng_list):
    # rng_list is a list of 2 uniform (between 0,1) random numbers
    
    team_1_win_prob = win_prob(SF_list[0], SF_list[1])
    team_3_win_prob = win_prob(SF_list[2], SF_list[3])
    
    F_pair = list()
    
    if rng_list[0] < team_1_win_prob:
        F_pair.append(SF_list[0])
    else:
        F_pair.append(SF_list[1])
    
    if rng_list[1] < team_3_win_prob:
        F_pair.append(SF_list[2])
    else:
        F_pair.append(SF_list[3])
    
    return(F_pair)

In [8]:
def pred_Winner(F_pair, rng_number):
    # rng_number is a uniform (between 0,1) random number
    
    team_1_win_prob = win_prob(F_pair[0], F_pair[1])
      
    if rng_number < team_1_win_prob:
        return(F_pair[0])
    else:
        return(F_pair[1])

In [9]:
num_trials = 1000

In [10]:
rng = default_rng(seed=123)
rng_values = rng.uniform(size = (num_trials,num_matches))

In [11]:
winners_list = list()
for i in range(num_trials):
    SF_list = pred_SF_list(QF_list, rng_values[i,0:4]) 
    F_pair = pred_F_pair(SF_list, rng_values[i,4:6])
    winner = pred_Winner(F_pair, rng_values[i,6])
    winners_list.append(winner)

In [12]:
winners_series = pd.Series(winners_list)

In [13]:
winners_series.value_counts() / num_trials

BRA    0.246
ARG    0.168
FRA    0.156
ENG    0.141
POR    0.112
NED    0.084
CRO    0.059
MAR    0.034
dtype: float64